# G04: Finding Circuits — The IOI Case Study

**Prerequisites:** G02 (cache), G03 (attention heads, direct logit attribution).

**What you'll learn:**
- How to set up an interpretability experiment with a clear task and metric
- Layer-level activation patching: which layers matter?
- Head-level activation patching: which specific heads implement the behavior?
- Ablation experiments: are those heads necessary?
- How to inspect and interpret the discovered circuit
- The full IOI circuit mechanism

This is the tutorial where everything comes together. We'll reverse-engineer the circuit that GPT-2 uses to solve a specific natural language task, following the same methodology as Wang et al. (2023).

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.datasets import make_ioi_dataset, to_tokens
from irtk.patching import (
    patch_by_layer, patch_by_head, ablate_heads,
    make_logit_diff_metric,
)
from irtk.circuits import direct_logit_attribution, residual_stream_attribution

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"GPT-2 Small: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads/layer")

## The Scientific Method for Circuits

Mechanistic interpretability is, at its core, a science. We form hypotheses about how a model computes a behavior, and then we test those hypotheses with interventions. The methodology we'll follow has five steps:

1. **Define a task** with clear correct/incorrect behavior. We need something where we can say unambiguously whether the model "got it right."
2. **Define a metric** that captures the behavior numerically. A single number that goes up when the model behaves correctly and down when it doesn't.
3. **Locate** the responsible components via patching. Which parts of the model are actually doing the work?
4. **Verify** via ablation and direct inspection. Are those components *necessary*, or just correlated with the behavior?
5. **Understand** the mechanism by examining component interactions. How do the pieces fit together?

This is the same methodology used in the landmark IOI paper (Wang et al., 2023, *Interpretability in the Wild*). Let's walk through it step by step.

## Step 1 — The IOI Task

**Indirect Object Identification (IOI)** is one of the most-studied tasks in mechanistic interpretability. Consider this sentence:

> "When **Mary** and **John** went to the store, **Mary** gave a gift to ___"

Humans immediately know the answer is **John**. But notice what the model has to do:

1. It sees two names: Mary and John.
2. It notices that **Mary** appeared again as the subject of "gave" — she's the one doing the giving.
3. It infers that the person being given *to* must be the **other** name — John.

In the IOI terminology:
- **IO** (Indirect Object) = John — the correct answer
- **S** (Subject) = Mary — the repeated name, which is the *wrong* answer

This is a great task for interpretability because:
- The model needs to do real computation (not just memorize)
- There's a clear correct answer and a clear wrong answer
- The wrong answer is a plausible alternative (both are names that appeared in the prompt)
- GPT-2 actually gets it right most of the time!

To create a **corrupted** baseline, we replace the IO name with a random name. This destroys the signal the model needs — with three different names, there's no clear "other name" to predict.

In [ ]:
# Generate IOI prompts with matched clean/corrupted pairs
dataset = make_ioi_dataset(n_prompts=20, tokenizer=tokenizer, seed=42)

print(f"Generated {len(dataset.clean_prompts)} prompt pairs\n")

for i in range(4):
    io = dataset.io_names[i]
    s = dataset.s_names[i]
    print(f"Example {i + 1}:")
    print(f"  Clean:     {dataset.clean_prompts[i]}")
    print(f"  Corrupted: {dataset.corrupted_prompts[i]}")
    print(f"  Correct answer (IO): {io!r}  (token {dataset.answer_tokens[i]})")
    print(f"  Wrong answer (S):    {s!r}  (token {dataset.wrong_tokens[i]})")
    print()

## Step 2 — The Metric

We need a single number that captures how well the model does on IOI. The standard choice is the **logit difference**:

$$\text{logit\_diff} = \text{logit}(\text{IO}) - \text{logit}(\text{S})$$

at the final token position (where the model makes its prediction).

- **Positive** logit diff → the model prefers IO (correct!)
- **Zero** → the model is indifferent between IO and S
- **Negative** → the model prefers S (wrong!)

Why logit difference and not accuracy? Because accuracy is binary — the model either gets it right or wrong. Logit difference is continuous, so we can see *how much* an intervention changes the model's preference. A head that shifts the logit diff by 2.0 is more important than one that shifts it by 0.1, even if both still leave the model correct.

In [ ]:
# Pick one example to work with throughout the tutorial
idx = 0
clean_tokens = to_tokens(dataset.clean_prompts[idx], tokenizer=tokenizer)
corrupted_tokens = to_tokens(dataset.corrupted_prompts[idx], tokenizer=tokenizer)
answer_token = dataset.answer_tokens[idx]
wrong_token = dataset.wrong_tokens[idx]

# Create the metric function: logit(IO) - logit(S) at the last position
metric = make_logit_diff_metric(correct_token=answer_token, wrong_token=wrong_token)

# Run the model on both clean and corrupted inputs
clean_logits = model(clean_tokens)
corrupted_logits = model(corrupted_tokens)

clean_ld = metric(clean_logits)
corrupted_ld = metric(corrupted_logits)

print(f"Prompt: {dataset.clean_prompts[idx]}")
print(f"Correct answer (IO): {tokenizer.decode([answer_token])!r}")
print(f"Wrong answer (S):    {tokenizer.decode([wrong_token])!r}")
print()
print(f"Clean logit diff:     {clean_ld:+.3f}  (positive = correct!)")
print(f"Corrupted logit diff: {corrupted_ld:+.3f}")
print(f"Gap:                  {clean_ld - corrupted_ld:.3f}")
print()
print("The model confidently prefers the IO on clean input, but not on corrupted input.")
print("Our goal: find WHICH components create this gap.")

## Step 3 — Layer-Level Patching

Now we start the detective work. The core idea behind **activation patching** is simple but powerful:

1. Run the model on the **clean** input (where it behaves correctly).
2. At one specific layer, **swap in** the residual stream from the **corrupted** run.
3. Measure the metric. If it drops toward the corrupted baseline, that layer was important.

Think of it like a car engine: if you swap in a faulty spark plug and the engine sputters, you know that spark plug matters. We're doing the same thing, one layer at a time.

This is a **causal** method — it tells us what the model actually *uses* for the computation, not just what correlates with the output. That's the key advantage over purely observational approaches.

In [ ]:
# Patch residual stream at each layer: clean run with corrupted layer swapped in
layer_results = patch_by_layer(model, clean_tokens, corrupted_tokens, metric)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(model.cfg.n_layers), layer_results, color='steelblue', edgecolor='black', linewidth=0.5)
ax.axhline(y=clean_ld, color='green', linestyle='--', alpha=0.7, label=f'Clean baseline ({clean_ld:.2f})')
ax.axhline(y=corrupted_ld, color='red', linestyle='--', alpha=0.7, label=f'Corrupted baseline ({corrupted_ld:.2f})')
ax.set_xlabel("Layer")
ax.set_ylabel("Logit Diff After Patching")
ax.set_title("Layer-Level Activation Patching\n(Corrupted residual stream swapped into clean run)")
ax.legend()
ax.set_xticks(range(model.cfg.n_layers))
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Identify the most impactful layers
layer_impact = clean_ld - layer_results  # positive = patching hurt performance
print("Most impactful layers (biggest drop when patched with corrupted activations):")
for l in np.argsort(layer_impact)[::-1][:5]:
    print(f"  Layer {l}: logit diff dropped by {layer_impact[l]:.3f}")

### Interpreting the Layer Results

Look at the bar chart. Each bar shows what happens to the logit difference when you corrupt that layer's residual stream. Bars near the green line mean "this layer doesn't matter much" — corrupting it barely changes the output. Bars near the red line mean "this layer is critical" — corrupting it destroys the IOI behavior.

You should see that a cluster of **early-to-mid layers** have the biggest impact. This makes sense intuitively: early layers process the input names and detect which name is repeated, mid layers route this information, and late layers use it to produce the output.

But layer-level patching is a blunt instrument. Each layer contains 12 attention heads plus an MLP — we need to zoom in to find the actual circuit components.

## Step 3b — Head-Level Patching

Now we apply the same logic at a finer granularity. Instead of patching an entire layer's residual stream, we patch each **individual attention head's output** (`z` — the pre-projection output) independently.

With 12 layers × 12 heads = 144 heads in GPT-2 Small, we get a 12×12 grid showing exactly which heads the IOI behavior depends on.

This is where the circuit starts to reveal itself.

In [ ]:
# Patch each head's z output independently
head_results = patch_by_head(model, clean_tokens, corrupted_tokens, metric)

# Visualize as a heatmap
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(head_results, aspect='auto', cmap='RdBu_r',
               vmin=corrupted_ld, vmax=clean_ld)

ax.set_xlabel("Head", fontsize=12)
ax.set_ylabel("Layer", fontsize=12)
ax.set_title("Head-Level Activation Patching\n"
             "(Color = logit diff after patching this head with corrupted activation)",
             fontsize=13)
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
plt.colorbar(im, ax=ax, label="Logit Diff")

# Annotate heads with large impact
head_impact = clean_ld - head_results
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        if abs(head_impact[layer, head]) > 0.5:
            ax.text(head, layer, f"{head_impact[layer, head]:+.1f}",
                    ha='center', va='center', fontsize=7, fontweight='bold',
                    color='white' if abs(head_impact[layer, head]) > 1.5 else 'black')

plt.tight_layout()
plt.show()

# List the top 10 most impactful heads
print("Top 10 heads by patching impact:")
print("(Positive = patching this head with corrupted activation reduces correct behavior)")
print("=" * 55)

flat_impact = head_impact.flatten()
sorted_indices = np.argsort(np.abs(flat_impact))[::-1]
for i in range(10):
    flat_idx = sorted_indices[i]
    layer = flat_idx // model.cfg.n_heads
    head = flat_idx % model.cfg.n_heads
    print(f"  L{layer}H{head}: impact = {head_impact[layer, head]:+.3f}")

### The IOI Circuit Emerges

Look at the heatmap. Most heads are near the green (clean) baseline — patching them barely matters. But a handful of heads stand out dramatically. These fall into distinct functional groups:

- **Name Mover Heads** (large *positive* impact — corrupting them reduces the logit diff): These heads are directly responsible for promoting the IO name at the output. In the full IOI circuit, heads like L9H9 and L10H0 are classic name movers. They attend from the final token position to the IO name position and "copy" the name to the output.

- **Negative Name Movers** (large *negative* impact — corrupting them actually *increases* the logit diff): These heads suppress the correct answer! They act as a counterbalance, possibly implementing a form of "backup" behavior or competition. The circuit uses these to calibrate its confidence.

- **S-Inhibition Heads** (mid-layer heads with positive impact): These attend to positions containing the subject name (S) and write inhibitory signals that prevent name movers from copying S instead of IO.

Wang et al. (2023) identified ~26 heads in the full IOI circuit, organized into these functional roles plus earlier "duplicate token" and "induction" heads that detect which name is repeated.

## Step 4 — Ablation Verification

Patching and ablation answer different questions:

- **Patching** asks: "What happens if this component receives *wrong* information?" (sufficiency test — is the component sufficient to change behavior?)
- **Ablation** asks: "What happens if this component is *removed entirely*?" (necessity test — is the component necessary?)

A head can show up in patching but not ablation if its contribution is redundant — other heads can compensate when it's removed. Conversely, a head might show up in ablation but not patching if it's important but receives the same input from both clean and corrupted runs.

Let's zero-ablate each head (set its output to zero) and see which heads are truly necessary.

In [ ]:
# Zero-ablate each head independently and measure the effect
ablation_results = ablate_heads(model, clean_tokens, metric, method="zero")

ablation_impact = clean_ld - ablation_results

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Left: patching impact
im0 = axes[0].imshow(head_impact, aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel("Head")
axes[0].set_ylabel("Layer")
axes[0].set_title("Patching Impact\n(corrupted swap → how much behavior changes)")
axes[0].set_xticks(range(model.cfg.n_heads))
axes[0].set_yticks(range(model.cfg.n_layers))
plt.colorbar(im0, ax=axes[0], label="LD Reduction")

# Right: ablation impact
im1 = axes[1].imshow(ablation_impact, aspect='auto', cmap='RdBu_r')
axes[1].set_xlabel("Head")
axes[1].set_ylabel("Layer")
axes[1].set_title("Zero Ablation Impact\n(head removed → how much behavior changes)")
axes[1].set_xticks(range(model.cfg.n_heads))
axes[1].set_yticks(range(model.cfg.n_layers))
plt.colorbar(im1, ax=axes[1], label="LD Reduction")

for ax_i in axes:
    data = head_impact if ax_i == axes[0] else ablation_impact
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            if abs(data[layer, head]) > 0.5:
                ax_i.text(head, layer, f"{data[layer, head]:+.1f}",
                          ha='center', va='center', fontsize=6, fontweight='bold')

plt.suptitle("Patching vs. Ablation: Two Views of the Same Circuit", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Top heads by ablation impact
print("Top 10 heads by ablation impact (most necessary for IOI):")
flat = ablation_impact.flatten()
for i in np.argsort(np.abs(flat))[::-1][:10]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: ablation impact = {ablation_impact[l, h]:+.3f}")

### Inspecting the Circuit: What Do These Heads Actually Do?

We now know *which* heads matter. The next question is *what* they do. The most direct way to understand an attention head is to look at its attention pattern: where does it attend from, and where does it attend to?

For IOI, we expect:
- **Name mover heads** attend from the **final token** (where the prediction happens) to the **IO name position** (to copy the correct name to the output).
- **S-inhibition heads** attend from the **final token** to the **S name position** (to gather information about which name to suppress).
- **Duplicate token heads** attend from the **second occurrence of S** to the **first occurrence of S** (to detect the repetition).

In [ ]:
# Run with cache to get attention patterns
logits, cache = model.run_with_cache(clean_tokens)
token_labels = [tokenizer.decode([int(t)]) for t in clean_tokens]

# Select the top 4 most impactful heads (by absolute patching impact)
flat_abs_impact = np.abs(head_impact).flatten()
top_4_flat = np.argsort(flat_abs_impact)[::-1][:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, flat_idx in zip(axes.flatten(), top_4_flat):
    layer = flat_idx // model.cfg.n_heads
    head = flat_idx % model.cfg.n_heads
    impact_val = head_impact[layer, head]

    # Get attention pattern for this head
    pattern = np.array(cache[("pattern", layer)])[head]  # [q_pos, k_pos]

    im = ax.imshow(pattern, cmap='Blues', vmin=0, vmax=pattern.max())

    role = "Name Mover" if impact_val > 0.5 else ("Negative Mover" if impact_val < -0.5 else "Other")
    ax.set_title(f"L{layer}H{head} — {role}\n(patching impact: {impact_val:+.2f})", fontsize=11)

    n_tokens = len(token_labels)
    if n_tokens <= 20:
        ax.set_xticks(range(n_tokens))
        ax.set_xticklabels(token_labels, rotation=90, fontsize=7)
        ax.set_yticks(range(n_tokens))
        ax.set_yticklabels(token_labels, fontsize=7)
    ax.set_xlabel("Key (attending to)")
    ax.set_ylabel("Query (attending from)")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Attention Patterns of the Most Impactful Heads", fontsize=14)
plt.tight_layout()
plt.show()

print("Look at the last row of each pattern (= what the final position attends to).")
print("Name movers should attend heavily to the IO name token.")
print("Negative movers may attend to the S name token.")

## The Full Circuit Mechanism

Putting all the pieces together, here's how GPT-2 solves IOI. This is the circuit that Wang et al. (2023) reverse-engineered:

### Phase 1: Duplicate Token Detection (Early Layers)
In layers 0–3, **duplicate token heads** detect that the subject name (S) appears twice in the prompt. These heads attend from the second occurrence of S to the first occurrence, effectively flagging "this name is repeated."

### Phase 2: S-Inhibition (Mid Layers)
In layers 7–8, **S-inhibition heads** take the duplicate-token signal and convert it into an inhibitory signal. They attend to the position of S and write information to the residual stream that says, essentially, "don't output this name." This is the crucial routing step.

### Phase 3: Name Moving (Late Layers)
In layers 9–11, **name mover heads** attend from the final position to name positions. Thanks to the S-inhibition signal, they preferentially attend to the IO name (which was *not* inhibited) and copy it to the output position. This directly increases the logit of the IO token.

### The Negative Name Movers
Some heads (the **negative name movers**) actively push *against* the IO token. Wang et al. hypothesize these are part of a calibration mechanism — the model hedges its bets, and the net effect is still strongly positive for IO.

### The Key Insight
The circuit is **compositional**: no single head can solve IOI alone. The early heads produce a signal that the mid heads transform, which the late heads use. This is exactly the kind of multi-step computation that makes transformers powerful — and that makes them hard to interpret without causal methods like activation patching.

In [ ]:
# Residual stream attribution: decompose the IO logit into per-component contributions
# This gives us a different view — not causal intervention, but direct contribution
contrib = residual_stream_attribution(model, cache, answer_token)

# Sort by absolute contribution
sorted_components = sorted(contrib.items(), key=lambda x: abs(x[1]), reverse=True)

# Plot the top 20 contributors
top_n = 20
names = [c[0] for c in sorted_components[:top_n]]
values = [c[1] for c in sorted_components[:top_n]]
colors = ['forestgreen' if v > 0 else 'firebrick' for v in values]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(top_n), values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Contribution to IO Logit")
ax.set_title(f"Residual Stream Attribution for {tokenizer.decode([answer_token])!r}\n"
             "(Decomposition of the IO token's logit into per-component contributions)")
ax.axvline(x=0, color='black', linewidth=0.8)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Green bars = components that promote the IO token (helpful for IOI).")
print("Red bars = components that suppress the IO token.")
print()
print("Compare with the patching results above — the same heads should appear.")
print("This is a consistency check: two different methods pointing to the same circuit.")

## Step 5 — Averaging Over Examples

Everything so far has been on a single IOI prompt. But individual examples can be noisy — maybe a head only matters on this particular sentence because of the specific token positions. To get a robust picture of the circuit, we should average our patching results over many examples.

To make results comparable across examples (which may have different baseline logit diffs), we **normalize**: we report the fraction of the clean logit diff that survives after patching. A value of 1.0 means the head's patching had no effect; a value near 0.0 means the head is critical.

In [ ]:
n_examples = min(10, len(dataset.clean_prompts))
all_head_results = []

for i in range(n_examples):
    ct = to_tokens(dataset.clean_prompts[i], tokenizer=tokenizer)
    crt = to_tokens(dataset.corrupted_prompts[i], tokenizer=tokenizer)
    m = make_logit_diff_metric(
        correct_token=dataset.answer_tokens[i],
        wrong_token=dataset.wrong_tokens[i],
    )

    # Need same-length sequences for fair comparison
    if ct.shape[0] != crt.shape[0]:
        continue

    hr = patch_by_head(model, ct, crt, m)
    clean_ld_i = m(model(ct))
    corrupted_ld_i = m(model(crt))

    # Normalize: fraction of clean logit diff preserved
    gap = clean_ld_i - corrupted_ld_i
    if abs(gap) > 0.1:  # skip degenerate cases
        normalized = (hr - corrupted_ld_i) / gap
        all_head_results.append(normalized)

print(f"Averaged over {len(all_head_results)} valid examples.")

if all_head_results:
    avg_results = np.mean(all_head_results, axis=0)

    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(avg_results, aspect='auto', cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_xlabel("Head", fontsize=12)
    ax.set_ylabel("Layer", fontsize=12)
    ax.set_title(f"Average Head Patching Results ({len(all_head_results)} examples)\n"
                 "(1.0 = no effect, 0.0 = completely corrupted behavior)",
                 fontsize=13)
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    plt.colorbar(im, ax=ax, label="Fraction of Clean Logit Diff")

    # Annotate heads that deviate strongly from 1.0
    deviation = np.abs(1.0 - avg_results)
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            if deviation[layer, head] > 0.15:
                ax.text(head, layer, f"{avg_results[layer, head]:.2f}",
                        ha='center', va='center', fontsize=6, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Most impactful heads (furthest from 1.0)
    print("\nHeads most affected by patching (averaged over examples):")
    flat_dev = deviation.flatten()
    for i in np.argsort(flat_dev)[::-1][:10]:
        l = i // model.cfg.n_heads
        h = i % model.cfg.n_heads
        print(f"  L{l}H{h}: avg normalized = {avg_results[l, h]:.3f}"
              f"  (deviation from 1.0 = {deviation[l, h]:.3f})")

## Key Takeaways

1. **Activation patching is a causal method.** It tells you what *matters* for a computation, not just what correlates with the output. This is the gold standard for mechanistic interpretability.

2. **Layer patching gives a coarse picture; head patching identifies specific components.** Always start broad and zoom in. Layer-level results tell you where to look; head-level results tell you what to find.

3. **Ablation tests necessity; patching tests sufficiency.** A component that shows up in both is a strong circuit member. Discrepancies between the two reveal redundancy and backup behavior.

4. **The IOI circuit uses ~26 heads organized into functional groups.** Name movers, S-inhibition heads, duplicate token heads, and induction heads work together in a multi-step computation. No single head can solve the task alone.

5. **The methodology generalizes.** The five-step process — task → metric → locate → verify → understand — applies to *any* model behavior. Wang et al. used it for IOI; you can use it for whatever behavior interests you.

6. **Residual stream attribution provides a complementary view.** It decomposes the output logit into per-component contributions without interventions. When patching and attribution agree, you can be more confident in your circuit.

## Exercises

1. **Mean ablation vs. zero ablation.** Run `ablate_heads(model, clean_tokens, metric, method="mean")` and compare the results with zero ablation. Mean ablation replaces a head's output with its average activation rather than zero. Do the same heads show up? When might the two methods disagree?

2. **OV circuit of name movers.** The name mover heads copy names to the output. Use `direct_logit_attribution(model, cache, answer_token)` to verify that these heads contribute positively to the IO logit. Then check `direct_logit_attribution(model, cache, wrong_token)` — do negative name movers promote the *wrong* token?

3. **Longer prompts.** Generate IOI prompts with more names (e.g., "When Mary, Alice, and John went to the store, Mary gave a gift to ___"). Does the same circuit activate? Do new heads become important?

4. **Tracing the full path.** The duplicate token heads in layers 0–3 feed into the S-inhibition heads in layers 7–8. Can you verify this by patching early heads and measuring the effect on S-inhibition head attention patterns? (Hint: use `model.run_with_cache` with and without early-head ablation, then compare the mid-layer patterns.)

## Further Reading

- **Wang et al. 2023** — ["Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 Small"](https://arxiv.org/abs/2211.00593). The paper this tutorial is based on. They identified the full 26-head IOI circuit with detailed functional characterization of each component.

- **Conmy et al. 2023** — ["Towards Automated Circuit Discovery for Mechanistic Interpretability"](https://arxiv.org/abs/2304.14997). ACDC automates the process of finding circuits, scaling beyond manual patching to discover circuits algorithmically.

- **Next tutorial: G05** — We'll build on this foundation to explore more advanced circuit analysis techniques.